<a href="https://colab.research.google.com/github/jayx003/GenAI-Practice/blob/main/04_context_window.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Context Window: Why Chatbots Forget

Every model has a **context window limit** - the maximum number of tokens it can process in a single request.

This includes input + output tokens combined.

Let's see what happens when you exceed it!

In [1]:
from google import genai
from google.genai import types
import os
# from dotenv import load_dotenv

# load_dotenv(dotenv_path='./.env')
# API_KEY = os.environ["GEMINI_API_KEY"]

from google.colab import userdata
API_KEY = userdata.get('GEMINI_API_KEY')

client = genai.Client(api_key=API_KEY)

In [2]:
# Create a VERY long message to hit the limit
long_story = "Once upon a time, there was a programmer who loved Python. "*10  # Suceed


#long_story = "Once upon a time, there was a programmer who loved Python. "*100000  # ERROR

print(f"📏 Story length: {len(long_story):,} characters")
print(f"📏 Estimated tokens: ~{len(long_story) // 4:,}\n")

try:
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=long_story,
        config={"max_output_tokens": 100}
    )
    print("✅ Request succeeded!")
    print(f"📊 Tokens used: {response.usage_metadata.total_token_count:,}")
    print(f"\n💡 Gemini 3.1 Flash has a HUGE context window (~1M tokens)")
    print(f"   So this request fits comfortably!")
except Exception as e:
    print(f"❌ ERROR: {e}")
    print("\n💡 This is what happens when you exceed the context window!")

📏 Story length: 590 characters
📏 Estimated tokens: ~147

✅ Request succeeded!
📊 Tokens used: 228

💡 Gemini 3.1 Flash has a HUGE context window (~1M tokens)
   So this request fits comfortably!


### What is Context Window?

**Context Window = Maximum tokens in a single request**

Different models have different limits:
- **Gemini 2.5 Flash**: ~1 million tokens
- **GPT-4**: ~128K tokens
- **Claude 4**: ~200K tokens




![image-2.png](attachment:image-2.png)


### Context Window in Conversations

In chat applications, the context window includes:
- **All previous messages** (entire conversation history)
- **Current message**
- **Response**

As conversations grow, tokens accumulate!

In [3]:
messages = []

def chat(user_message):
    """Send message and track tokens"""
    messages.append(
        types.Content(role="user", parts=[types.Part(text=user_message)])
    )

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=messages
    )

    messages.append(
        types.Content(role="model", parts=[types.Part(text=response.text)])
    )

    # Show token usage
    total_tokens = response.usage_metadata.total_token_count
    print(f"🤖: {response.text}")
    print(f"📊 Total tokens used: {total_tokens} (includes ALL {len(messages)} messages)\n")

    return response.text

# Start conversation
print("👤: Hi! My name is Yash")
chat("Hi! My name is Yash")

print("👤: I'm 25 years old")
chat("I'm 25 years old")

print("👤: I love Python programming")
chat("I love Python programming")

print("👤: I work as a software engineer")
chat("I work as a software engineer")

print("👤: What's my name?")
chat("What's my name?")

👤: Hi! My name is Yash
🤖: Hi Yash! It's nice to meet you. How are you doing today? Is there anything I can help you with?
📊 Total tokens used: 33 (includes ALL 2 messages)

👤: I'm 25 years old
🤖: That’s a great age, Yash! You're in your mid-twenties, which is often a really interesting time for career growth, exploring new hobbies, or just figuring out what you want your next few years to look like.

How is life treating you at 25? Are you currently working, studying, or focused on any particular projects or goals?
📊 Total tokens used: 119 (includes ALL 4 messages)

👤: I love Python programming
🤖: That’s awesome, Yash! Python is such a versatile and rewarding language to be into. Whether you're interested in web development, data science, automation, or AI, Python is pretty much the "Swiss Army knife" of the programming world.

What specifically do you like about it? Are you working on any cool projects right now, or are you still exploring different libraries and frameworks? 

Also, i

'Your name is **Yash**!'

### The Problem: Tokens Keep Growing

Notice how tokens increase with each message?

In a real long conversation:
- Message 1: 50 tokens
- Message 10: 500 tokens  
- Message 50: 2,500 tokens
- Message 100: 5,000 tokens
- Message 1000: 50,000 tokens

**Eventually, you'll hit the context window limit!**

### Solution: Remove Old Messages

When approaching the limit, remove oldest messages.

**This is why chatbots "forget"!**

In [5]:
MAX_MESSAGES = 6  # Keep only last 6 messages (3 exchanges)

def chat_with_limit(user_message):
    """Chat with context window management"""
    messages.append(
        types.Content(role="user", parts=[types.Part(text=user_message)])
    )

    # Remove old messages if too many
    if len(messages) > MAX_MESSAGES:
        removed = messages.pop(0)  # Remove oldest
        print(f"🗑️  Removed old message: {removed.parts[0].text[:50]}...\n")

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=messages
    )

    messages.append(
        types.Content(role="model", parts=[types.Part(text=response.text)])
    )

    print(f"🤖: {response.text}")
    print(f"📊 Messages in memory: {len(messages)}\n")

    return response.text

# Reset and try again
messages = []

print("👤: My favorite color is blue")
chat_with_limit("My favorite color is blue")

print("👤: I have a dog named Max")
chat_with_limit("I have a dog named Max")

print("👤: I live in Mumbai")
chat_with_limit("I live in Mumbai")

print("👤: I enjoy hiking")
chat_with_limit("I enjoy hiking")

# This will forget the first message!
print("👤: What's my favorite color?")
chat_with_limit("What's my favorite color?")

👤: My favorite color is blue
🤖: That's a great choice! Blue is such a classic and calming color. It reminds me of the ocean and a clear summer sky. 

Do you have a specific shade of blue that you like best, like navy, sky blue, or teal?
📊 Messages in memory: 2

👤: I have a dog named Max
🤖: Max is a great name for a dog! 

What kind of dog is he? Is he a goofy puppy, a lazy couch potato, or full of energy?
📊 Messages in memory: 4

👤: I live in Mumbai
🤖: Mumbai is such an incredible, vibrant city! There is so much energy there, from the bustling streets to the beautiful views along Marine Drive. 

How does Max like living in Mumbai? Does he have a favorite spot in the city or a favorite park for walks?
📊 Messages in memory: 6

👤: I enjoy hiking
🗑️  Removed old message: My favorite color is blue...

🤖: Hiking is such a great way to disconnect and enjoy nature! 

Since you live in Mumbai, do you ever get the chance to go hiking in the Western Ghats nearby? Places like Sanjay Gandhi Nationa

"Haha, I actually don't know what your favorite color is yet! You haven't mentioned it. \n\nIs it something earthy like green from all your hiking, or maybe something bright like Mumbai's sunsets? What is it?"

In [6]:
chat_with_limit("What is a special about Mango?")

🗑️  Removed old message: I have a dog named Max...

🤖: Oh, mangoes are practically royalty in India—especially in Mumbai! There are a few things that make them super special:

1. **The King of Fruits:** The Alphonso mango (locally called *Hapus*), which grows along the coast of Maharashtra near Ratnagiri, is world-famous for its incredible sweetness, rich flavor, and buttery texture. People *live* for Alphonso season in Mumbai!
2. **Cultural Obsession:** In India, mangoes aren't just fruit; they are an emotion. Summer in Mumbai is basically defined by the arrival of mangoes. 
3. **Versatility:** Whether you're eating them sliced fresh, blended into a thick *aamras*, made into milkshakes, or used in pickles, they are delicious in every possible way. 

Are you a big fan of mangoes? (And does Max ever get a little taste?)
📊 Messages in memory: 9



"Oh, mangoes are practically royalty in India—especially in Mumbai! There are a few things that make them super special:\n\n1. **The King of Fruits:** The Alphonso mango (locally called *Hapus*), which grows along the coast of Maharashtra near Ratnagiri, is world-famous for its incredible sweetness, rich flavor, and buttery texture. People *live* for Alphonso season in Mumbai!\n2. **Cultural Obsession:** In India, mangoes aren't just fruit; they are an emotion. Summer in Mumbai is basically defined by the arrival of mangoes. \n3. **Versatility:** Whether you're eating them sliced fresh, blended into a thick *aamras*, made into milkshakes, or used in pickles, they are delicious in every possible way. \n\nAre you a big fan of mangoes? (And does Max ever get a little taste?)"

### Key Takeaways

1. **Context Window** = Maximum tokens in a single request (input + output)

2. **In conversations**, tokens grow with each message

3. **Must remove old messages** when approaching limit

4. **This causes "forgetting"** - AI loses old context

5. **Exceeding limit = Error** - Request will fail

### Real-World Solutions

- **Summarize** old messages instead of removing
- **Store** important info separately (database)
- **Prioritize** recent messages
- **Use RAG** (Retrieval Augmented Generation) for long-term memory
- **Monitor token usage** and manage proactively